<a href="https://colab.research.google.com/github/aswithapatel-0518/ML_LABPROGRAMS/blob/main/mlex_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Experiment 10 : Full Metrics Suite
# ============================================================

!pip -q install statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.calibration import calibration_curve

from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    matthews_corrcoef,
    f1_score,
    RocCurveDisplay
)

from statsmodels.stats.contingency_tables import mcnemar


# ============================================================
# 1. LOAD PLACEMENT DATASET
# ============================================================

uploaded = files.upload()

filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Loaded file:", filename)
print("Dataset shape:", df.shape)


# ============================================================
# 2. SELECT FEATURES
# ============================================================

F = [
    "CGPA",
    "AttendancePercent",
    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",
    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore"
]


# ============================================================
# 3. PREPARE FEATURES
# ============================================================

X = df[F].copy()

# Handle missing values
X = X.fillna(X.median())


# ============================================================
# 4. PREPARE TARGET
# ============================================================

y = (
    df["PlacementStatus"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "placed": 1,
        "not placed": 0,
        "not_placed": 0,
        "notplaced": 0
    })
)


# Remove rows with invalid target values
valid = y.notna()

X = X.loc[valid].reset_index(drop=True)

y = y.loc[valid].astype(int).reset_index(drop=True)


print("\nValid samples:", len(y))

print("\nPlacement distribution:")
print(y.value_counts())


# ============================================================
# 5. TRAIN / TEST SPLIT
# ============================================================

Xtr, Xte, ytr, yte = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


print("\nTraining samples:", len(Xtr))
print("Testing samples :", len(Xte))


# ============================================================
# 6. DEFINE MODELS
# ============================================================

models = {

    "LogReg":
        make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=1000
            )
        ),

    "RF":
        RandomForestClassifier(
            n_estimators=300,
            random_state=0,
            n_jobs=-1
        ),

    "NB":
        make_pipeline(
            StandardScaler(),
            GaussianNB()
        )
}


# ============================================================
# 7. TRAIN MODELS AND PREDICT
# ============================================================

pr = {}
pd_ = {}


for name, model in models.items():

    print(
        f"Training {name}..."
    )

    model.fit(
        Xtr,
        ytr
    )

    pr[name] = model.predict_proba(
        Xte
    )[:, 1]

    pd_[name] = model.predict(
        Xte
    )


print("\nAll models trained.")


# ============================================================
# 8. FULL METRICS
# ============================================================

rows = []


for name in models:

    tn, fp, fn, tp = confusion_matrix(
        yte,
        pd_[name]
    ).ravel()


    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0
    )


    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )


    rows.append({

        "model": name,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1_score(
                yte,
                pd_[name]
            ),

        "MCC":
            matthews_corrcoef(
                yte,
                pd_[name]
            ),

        "ROC_AUC":
            roc_auc_score(
                yte,
                pr[name]
            ),

        "PR_AUC":
            average_precision_score(
                yte,
                pr[name]
            ),

        "brier":
            brier_score_loss(
                yte,
                pr[name]
            )
    })


results = pd.DataFrame(rows)


# ============================================================
# 9. DISPLAY RESULTS
# ============================================================

print(
    "\n========== FULL METRICS ==========\n"
)

print(
    results
    .round(4)
    .to_string(index=False)
)


# ============================================================
# 10. CONFUSION MATRICES
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 4)
)


for ax, name in zip(
    axes,
    models.keys()
):

    cm = confusion_matrix(
        yte,
        pd_[name]
    )

    ax.imshow(cm)

    ax.set_title(
        f"{name} Confusion Matrix"
    )

    ax.set_xlabel(
        "Predicted"
    )

    ax.set_ylabel(
        "Actual"
    )

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    for i in range(2):

        for j in range(2):

            ax.text(
                j,
                i,
                cm[i, j],
                ha="center",
                va="center"
            )


plt.tight_layout()
plt.show()


# ============================================================
# 11. ROC CURVES
# ============================================================

plt.figure(figsize=(8, 6))

ax = plt.gca()


for name, model in models.items():

    RocCurveDisplay.from_estimator(
        model,
        Xte,
        yte,
        ax=ax,
        name=name
    )


plt.plot(
    [0, 1],
    [0, 1],
    'k--',
    label="Random"
)

plt.title(
    "ROC Curves"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.legend()

plt.show()


# ============================================================
# 12. CALIBRATION / RELIABILITY CURVES
# ============================================================

plt.figure(figsize=(8, 6))

plt.plot(
    [0, 1],
    [0, 1],
    'k--',
    label="Perfect calibration"
)


for name in models:

    fraction_positive, mean_predicted = calibration_curve(
        yte,
        pr[name],
        n_bins=10,
        strategy="quantile"
    )

    plt.plot(
        mean_predicted,
        fraction_positive,
        'o-',
        label=name
    )


plt.xlabel(
    "Predicted Probability"
)

plt.ylabel(
    "Observed Frequency"
)

plt.title(
    "Reliability / Calibration Curve"
)

plt.legend()

plt.show()


# ============================================================
# 13. McNEMAR TEST
# ============================================================

print(
    "\n========== McNEMAR TEST =========="
)


for a, b in [
    ("LogReg", "RF"),
    ("LogReg", "NB"),
    ("RF", "NB")
]:

    ca = (
        pd_[a] == yte.values
    )

    cb = (
        pd_[b] == yte.values
    )


    table = [
        [
            np.sum(ca & cb),
            np.sum(ca & ~cb)
        ],

        [
            np.sum(~ca & cb),
            np.sum(~ca & ~cb)
        ]
    ]


    test = mcnemar(
        table,
        exact=False,
        correction=True
    )


    p = test.pvalue


    print(
        f"{a} vs {b}: "
        f"p={p:.4f} -> ",
        "significant"
        if p < 0.05
        else "not significant"
    )


# ============================================================
# 14. FINAL BEST MODEL
# ============================================================

best_model = results.loc[
    results["ROC_AUC"].idxmax()
]

print(
    "\n========== BEST MODEL =========="
)

print(
    "Best model based on ROC-AUC:",
    best_model["model"]
)

print(
    "ROC-AUC:",
    round(best_model["ROC_AUC"], 4)
)

print(
    "PR-AUC:",
    round(best_model["PR_AUC"], 4)
)

print(
    "F1:",
    round(best_model["f1"], 4)
)

print(
    "MCC:",
    round(best_model["MCC"], 4)
)